# Enhanced S3 to COG Converter with Automatic AWS Authentication

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Support for multiple AWS authentication methods**

Author: Kyle Lesinger (Enhanced version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm
import re

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")


✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes,
    get_available_memory_mb

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked,
)

# Import the FIXED improved version that prevents striping
from convert_utilities_improved_fixed import (
    convert_to_proper_CRS_and_cogify_improved_fixed,
)
    
print("✅ Custom modules imported successfully!")
print("   Using FIXED improved converter to prevent striping issues")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Fixed conversion utilities loaded - striping issue resolved!

Key fix: Maintains consistent chunk grid throughout processing
Use convert_to_proper_CRS_and_cogify_improved_fixed() for large files
✅ Custom modules imported successfully!
   Using FIXED improved converter to prevent striping issues
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


## Monitoring Memory Usage

# Useful links

[drcs_activations OLD Directory](https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/)

[VEDA docs for file naming conventions](https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html)

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:
EVENT_NAME = '202504_SevereWx_US'
#old name
#under drcs_activations
PRODUCT_NAME = 'sentinel2'


PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Standard chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

# Enhanced configuration for large Sentinel-2 files (3-10GB)
LARGE_FILE_CONFIG = {
    "default_chunk_size": 256,       # Smaller chunks for large files
    "memory_limit_mb": 250,          # Conservative memory limit
    "aggressive_gc": True,           # Force gc after each band
    "single_band_mode": True,       # Sentinel-2 has multiple bands but manageable
    "use_streaming": False,           # Stream from S3 to avoid download
    "cleanup_immediate": True,       # Delete temp files ASAP
    "adaptive_chunks": False,         # Dynamic chunk sizing
    "max_retries": 3,               # Retry on failure
    "min_chunk_size": 256,
    "max_chunk_size": 1024          # Cap at 1024 for Sentinel-2
}

# Ultra-large file config (for files > 10GB)
ULTRA_LARGE_CONFIG = {
    "default_chunk_size": 1024,       # Very small chunks
    "memory_limit_mb": 150,          # Very conservative memory
    "aggressive_gc": True,
    "single_band_mode": True,        # Process bands one at a time
    "use_streaming": False,
    "cleanup_immediate": True,
    "adaptive_chunks": False,
    "max_retries": 5,
    "min_chunk_size": 1024,
    "max_chunk_size": 1024
}

## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name='nasa-disasters', verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name='nasa-disasters', verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, 'nasa-disasters', PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 55 .tif files in the S3 bucket.


['drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2A_MNDWI_20250408_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2A_NDVI_20250322_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2A_NDVI_20250408_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2A_trueColor_20250322_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2A_trueColor_20250408_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2B_NDVI_20250322_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2B_trueColor_20250322_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2C_MNDWI_20250409_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2C_NDVI_20250409_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2C_trueColor_20250409_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/LZK_S2B_MNDWI_20250407_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/LZK_S2B_NDVI_202

## ⚠️ IMPORTANT: Large File Processing Notes

### Striping Issue Fix
This notebook includes a **FIXED** version that prevents striping issues in ultra-large files (>7GB).

**The Problem:** 
- Previous versions would reduce chunk size during processing when memory got high
- This caused misaligned chunks, creating horizontal stripes in the output

**The Solution:**
- Files >7GB: Use **128x128 FIXED chunks** throughout entire processing
- Files 3-7GB: Use **256x256 FIXED chunks** 
- Files <3GB: Standard adaptive processing

### Temp File Error Fix
If you see errors like `[Errno 2] No such file or directory: '/tmp/tmpXXX.tif'`:
- The notebook will automatically retry with an alternative temp directory
- This happens when /tmp runs out of space or has permission issues

### Processing Times
With the fixed chunk approach:
- 7.4 GB file: ~11-15 minutes (128x128 chunks)
- 3-7 GB file: ~5-10 minutes (256x256 chunks)
- <3 GB file: ~2-5 minutes (adaptive chunks)

In [7]:
import re
import os
import tempfile

def simple_process_files_improved(keys, filter_str, rename_func, target_dir, EVENT_NAME, 
                                  use_improved=True, file_size_threshold_gb=3):
    """
    Enhanced wrapper with FIXED converter that prevents striping issues.
    
    IMPORTANT FIX: This version prevents the striping issue that occurred with 
    ultra-large files (>7GB) by maintaining consistent chunk sizes throughout processing.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'NDVI')
            - Regex pattern object 
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-2/NDVI")
        EVENT_NAME: Event name
        use_improved: Whether to use the improved converter (default: True)
        file_size_threshold_gb: Threshold in GB to switch to large file config (default: 3)
    
    Returns:
        Processing results DataFrame
    
    Notes:
        - Files >10GB: Uses 128x128 FIXED chunks, single-band processing
        - Files 3-10GB: Uses 256x256 FIXED chunks
        - Files <3GB: Standard processing with adaptive chunks
    """
    # Set temp directory to a reliable location with more space
    os.environ['TMPDIR'] = '/tmp'
    tempfile.tempdir = '/tmp'
    
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        pattern = re.compile(filter_str[2:-1])
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        filtered_files = [i for i in keys if filter_str in i]
    
    # 2. Test renaming (show first 5 examples)
    print(f"Testing filenames:")
    for f in filtered_files[:5]:
        print(f"  {rename_func(f, EVENT_NAME)}")
    if len(filtered_files) > 5:
        print(f"  ... and {len(filtered_files) - 5} more files")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print(f"🚀 Processing {len(filtered_files)} Files")
    if use_improved:
        print("   ✅ Using FIXED improved converter (prevents striping)")
        print("   📝 Ultra-large files (>7GB) will use 128x128 fixed chunks")
        print("   📝 Large files (3-7GB) will use 256x256 fixed chunks")
    else:
        print("   Using standard chunked converter")
    print("="*50)
    
    def get_file_size_gb(s3_client, bucket, key):
        """Get file size in GB from S3."""
        try:
            response = s3_client.head_object(Bucket=bucket, Key=key)
            size_gb = response['ContentLength'] / (1024**3)
            return size_gb
        except:
            return 0
    
    def improved_converter_fixed(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, 
                                 s3_client, local_output_dir=None):
        """
        FIXED converter that prevents striping by maintaining consistent chunk sizes.
        
        The striping issue was caused by chunk_size being reduced during iteration,
        creating misaligned chunks. This version uses FIXED chunk sizes throughout.
        """
        # Check file size
        file_size_gb = get_file_size_gb(s3_client, BUCKET, name)
        
        # Select appropriate FIXED config based on file size
        if file_size_gb > 7:  # Adjusted threshold for ultra-large
            print(f"   📦 Ultra-large file ({file_size_gb:.1f} GB), using 128x128 FIXED chunks")
            chunk_config = {
                "default_chunk_size": 256,      # Small FIXED chunks
                "memory_limit_mb": 150,         # Very conservative memory
                "aggressive_gc": True,          # Force gc after each band
                "single_band_mode": True,       # Process bands one at a time
                "use_streaming": False,          # Stream from S3
                "cleanup_immediate": True,      # Delete temp files ASAP
                "adaptive_chunks": False,       # DISABLE adaptive - use fixed chunks
                "max_retries": 5,
                "show_progress": True,
                "enable_memory_monitoring": True,
                "min_chunk_size": 256,
                "max_chunk_size": 256          # Force fixed size
            }
        elif file_size_gb > file_size_threshold_gb:
            print(f"   📦 Large file ({file_size_gb:.1f} GB), using 256x256 FIXED chunks")
            chunk_config = {
                "default_chunk_size": 256,      # Medium FIXED chunks
                "memory_limit_mb": 250,         # Conservative memory
                "aggressive_gc": True,
                "single_band_mode": True,      # Can handle multiple bands
                "use_streaming": True,
                "cleanup_immediate": True,
                "adaptive_chunks": False,       # DISABLE adaptive - use fixed chunks
                "max_retries": 3,
                "show_progress": True,
                "enable_memory_monitoring": True,
                "min_chunk_size": 256,
                "max_chunk_size": 256          # Force fixed size
            }
        else:
            print(f"   📦 Standard file ({file_size_gb:.1f} GB), using adaptive chunks")
            chunk_config = CHUNK_CONFIG    # Standard config with adaptive chunks
        
        try:
            # Use the FIXED version that prevents striping
            return convert_to_proper_CRS_and_cogify_improved_fixed(
                name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, 
                s3_client, COG_PROFILE, local_output_dir, chunk_config
            )
        except FileNotFoundError as e:
            # Handle temp file errors
            if '/tmp/' in str(e):
                print(f"   [ERROR] Temporary file issue: {e}")
                print(f"   [RETRY] Attempting with alternative temp directory...")
                
                # Try with a different temp directory
                alt_temp_dir = os.path.join(os.getcwd(), 'temp_cog')
                os.makedirs(alt_temp_dir, exist_ok=True)
                tempfile.tempdir = alt_temp_dir
                
                # Retry with new temp settings
                return convert_to_proper_CRS_and_cogify_improved_fixed(
                    name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, 
                    s3_client, COG_PROFILE, local_output_dir, chunk_config
                )
            else:
                raise
    
    def standard_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, 
                          s3_client, local_output_dir=None):
        """Standard chunked converter for smaller files."""
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, 
            s3_client, COG_PROFILE, local_output_dir, chunk_config=CHUNK_CONFIG
        )
    
    # Select converter
    if use_improved:
        converter_func = improved_converter_fixed
    else:
        converter_func = standard_converter
    
    # Process files
    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=converter_func,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Keep the original function for backward compatibility
simple_process_files = simple_process_files_improved

print("✅ Updated processing function loaded with FIXED chunk handling")
print("   - Prevents striping issues in ultra-large files")
print("   - Handles temp file errors with fallback directory")

✅ Updated processing function loaded with FIXED chunk handling
   - Prevents striping issues in ultra-large files
   - Handles temp file errors with fallback directory


In [8]:
# For simplicity, let's use python list comprehension to return the files
# We may need to rename them in different ways for different products
# We will do a similar process later

## NOTE --- We can actually use these objects since they have the same path as the s3 files. We will call them again later

ndvi = [f for f in keys if "NDVI" in f]#NDVI
true = [f for f in keys if "true" in f]#true
mndwi = [f for f in keys if "MNDWI" in f]#MNDWI

## Configure bucket and paths (no need to create session manually)

In [9]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [10]:
def convert_date(date_str):
    """
    Convert to YYYY-MM-DD.
    
    Args:
        datetime_str: String like '20250731'
    
    Returns:
        String like '2025-07-31'
    """
    # Extract components
    year = date_str[0:4]
    month = date_str[4:6]
    day = date_str[6:8]
    
    # Format with dashes and colons, add Z for UTC
    return f"{year}-{month}-{day}"

# Test
date_str = '20250731'
result = convert_date(date_str)
print(result)

2025-07-31


In [11]:
def create_cog_filename(f, EVENT_NAME):
    """Create COG filename for NDVI files."""
    # Extract directory, filename, and extension
    directory, filename = os.path.split(f)
    stem, ext = os.path.splitext(filename)

    # Find all 8-digit date patterns
    dates = re.findall(r"\d{8}", stem)

    # Remove dates from the stem
    stem_clean = re.sub(r"_?\d{8}", "", stem)

    # Build new stem
    cog_filename = f"{EVENT_NAME}_{stem_clean}_{convert_date(dates[0])}_day.tif"
    return cog_filename


filter_str = 'NDVI'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:
  202504_SevereWx_US_JAN_S2A_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_JAN_S2A_NDVI_merged_2025-04-08_day.tif
  202504_SevereWx_US_JAN_S2B_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_JAN_S2C_NDVI_merged_2025-04-09_day.tif
  202504_SevereWx_US_LZK_S2B_NDVI_merged_2025-04-07_day.tif
  202504_SevereWx_US_LZK_S2C_NDVI_merged_2015-03-13_day.tif
  202504_SevereWx_US_LZK_S2C_NDVI_merged_2025-04-09_day.tif
  202504_SevereWx_US_MEG_S2A_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_MEG_S2A_NDVI_merged_2025-04-08_day.tif
  202504_SevereWx_US_MEG_S2B_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_MEG_S2C_NDVI_merged_2025-04-09_day.tif
  202504_SevereWx_US_OHX_S2A_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_OHX_S2A_NDVI_merged_2025-04-08_day.tif
  202504_SevereWx_US_OHX_S2B_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_PAH_S2A_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_PAH_S2A_NDVI_merged_2025-04-08_day.tif
  202504_SevereWx_U

In [12]:
def monitor_processing(results_df):
    """
    Monitor processing results and identify problematic files.
    
    Args:
        results_df: DataFrame with processing results
    """
    if results_df is None or results_df.empty:
        print("No results to monitor")
        return
        
    if 'status' in results_df.columns:
        # Check for failures
        failed = results_df[results_df['status'] == 'failed'] if 'status' in results_df.columns else pd.DataFrame()
        if not failed.empty:
            print("\n⚠️ Failed files:")
            for idx, row in failed.iterrows():
                print(f"  - {row.get('original_file', 'Unknown')}: {row.get('error', 'Unknown error')}")
        
        # Check for skipped files (already exist)
        skipped = results_df[results_df['status'] == 'skipped'] if 'status' in results_df.columns else pd.DataFrame()
        if not skipped.empty:
            print(f"\n✅ {len(skipped)} files skipped (already exist in S3)")
        
        # Successful files
        success = results_df[results_df['status'] == 'success'] if 'status' in results_df.columns else pd.DataFrame()
        if not success.empty:
            print(f"\n✅ {len(success)} files processed successfully")
    
    # Memory usage analysis if available
    if 'peak_memory_mb' in results_df.columns:
        print(f"\n📊 Memory Usage Statistics:")
        print(f"  Average: {results_df['peak_memory_mb'].mean():.1f} MB")
        print(f"  Maximum: {results_df['peak_memory_mb'].max():.1f} MB")
        print(f"  Minimum: {results_df['peak_memory_mb'].min():.1f} MB")
    
    # Processing time analysis if available
    if 'processing_time_s' in results_df.columns:
        total_time = results_df['processing_time_s'].sum()
        print(f"\n⏱️ Processing Time:")
        print(f"  Total: {total_time/60:.1f} minutes")
        print(f"  Average per file: {results_df['processing_time_s'].mean():.1f} seconds")

print("✅ Monitoring function defined")

✅ Monitoring function defined


There's three types of NDVI files in this list, so I will handle renaming all three with conditional statements:

In [13]:
# Monitor NDVI processing results
if 'results1' in locals():
    monitor_processing(results1)

# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 6
  - Total size: 45.13 GB

📁 Cached files (first 10):
  - drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2A_NDVI_20250322_merged.tif (7622.4 MB)
  - drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2C_NDVI_20250409_merged.tif (7615.3 MB)
  - drcs_activations/202504_SevereWx_US/sentinel2/LZK_S2B_NDVI_20250407_merged.tif (7970.4 MB)
  - drcs_activations/202504_SevereWx_US/sentinel2/MEG_S2A_NDVI_20250322_merged.tif (7518.0 MB)
  - drcs_activations/202504_SevereWx_US/sentinel2/MEG_S2C_NDVI_20250409_merged.tif (7518.0 MB)
  - drcs_activations/202504_SevereWx_US/sentinel2/SHV_S2B_NDVI_20250407_merged.tif (7969.5 MB)


(6, 48458468300)

In [ ]:
# Process additional NDVI files with improved processing (if needed)
# This cell can be used if you want to process NDVI files separately from Cell 18
# or if you need to reprocess with different settings
if ndvi and True:  # Set to True to enable this processing
    print("\n" + "="*50)
    print("🌿 Processing NDVI Files (Improved)")
    print("="*50)
    
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    # Process with improved converter
    ndvi_results = simple_process_files_improved(
        keys=keys, 
        filter_str='NDVI', 
        rename_func=create_cog_filename, 
        target_dir="Sentinel-2/NDVI", 
        EVENT_NAME=EVENT_NAME,
        use_improved=True,
        file_size_threshold_gb=3  # Adjust threshold as needed
    )
    
    all_files_processed = pd.concat([all_files_processed, ndvi_results], ignore_index=True)
    
    # Monitor results
    monitor_processing(all_files_processed)
    
    # Print overall summary
    print_batch_summary(all_files_processed)


🌿 Processing NDVI Files (Improved)
Testing filenames:
  202504_SevereWx_US_JAN_S2A_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_JAN_S2A_NDVI_merged_2025-04-08_day.tif
  202504_SevereWx_US_JAN_S2B_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_JAN_S2C_NDVI_merged_2025-04-09_day.tif
  202504_SevereWx_US_LZK_S2B_NDVI_merged_2025-04-07_day.tif
  ... and 17 more files
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202504_SevereWx_US/sentinel2
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-2/NDVI

🚀 Processing 22 Files
   ✅ Using FIXED improved converter (prevents striping)
   📝 Ultra-large files (>7GB) will use 128x128 fixed chunks
   📝 Large files (3-7GB) will use 256x256 fixed chunks
✅ Local output directory ready: output/202504_SevereWx_US

[1/22] Processing: drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2A_NDVI_20250322_merged.tif
   Output filename: 202504_SevereWx_US_JAN_S2A_NDVI_merged_2025-03-

Processing chunks:   0%|          | 0/31262 [00:00<?, ?chunks/s]

   [BAND 1/1] Processing...


Processing chunks:   1%|▏         | 447/31262 [00:05<05:21, 95.75chunks/s] 


   [MEMORY] High usage detected (588.7 MB), using memory-safe mode


Processing chunks: 100%|██████████| 31262/31262 [2:45:14<00:00,  3.15chunks/s]  


      Memory after band 1: 1103.2 MB
   [COGIFY] Preparing file for upload...


The following warnings were found:
- The file is greater than 512xH or 512xW, it is recommended to include internal overviews



   [COG] Reprojected file is already a valid COG!
   [COG] Reprojected file is already a valid COG, but rebuilding with overviews...
   [COG] Processing 3345.2 MB file...
   [COG] Using ZSTD compression with predictor=3 for float32 data
   [COG] Writing optimized COG...


In [ ]:
# Process MNDWI files with improved processing
if mndwi:
    print("\n" + "="*50)
    print("🌊 Processing MNDWI Files (Improved)")
    print("="*50)
    
    # Process with improved converter
    mndwi_results = simple_process_files_improved(
        keys=keys, 
        filter_str='MNDWI', 
        rename_func=create_cog_filename, 
        target_dir="Sentinel-2/MNDWI", 
        EVENT_NAME=EVENT_NAME,
        use_improved=True,
        file_size_threshold_gb=3  # MNDWI files might be large
    )
    
    # Monitor results
    monitor_processing(mndwi_results)
    
    # Print overall summary
    print_batch_summary(mndwi_results)

In [ ]:
# Process True Color files with improved processing
if true:
    print("\n" + "="*50)
    print("🎨 Processing True Color Files (Improved)")
    print("="*50)
    
    # Process with improved converter
    # True Color files are typically the largest, so use lower threshold
    true_results = simple_process_files_improved(
        keys=keys, 
        filter_str='trueColor', 
        rename_func=create_cog_filename, 
        target_dir="Sentinel-2/RGB", 
        EVENT_NAME=EVENT_NAME,
        use_improved=True,
        file_size_threshold_gb=2  # Lower threshold for RGB (usually larger files)
    )
    
    # Monitor results
    monitor_processing(true_results)
    
    # Print overall summary
    print_batch_summary(true_results)

In [ ]:
# Display final combined results
print(f"\n📊 Final Processing Results Summary:")
print("="*60)

# Combine all results if they exist
all_results = []
result_names = []

if 'results1' in locals():
    all_results.append(results1)
    result_names.append("NDVI")
    
if 'mndwi_results' in locals():
    all_results.append(mndwi_results)
    result_names.append("MNDWI")
    
if 'true_results' in locals():
    all_results.append(true_results)
    result_names.append("True Color")

if all_results:
    # Combine all DataFrames
    all_files_processed = pd.concat(all_results, ignore_index=True)
    
    print(f"Total files processed across all types: {len(all_files_processed)}")
    
    # Summary by type
    for name, df in zip(result_names, all_results):
        print(f"\n{name} Files:")
        print(f"  - Total: {len(df)}")
        if 'status' in df.columns:
            success = len(df[df['status'] == 'success'])
            failed = len(df[df['status'] == 'failed'])
            skipped = len(df[df['status'] == 'skipped'])
            print(f"  - Success: {success}")
            print(f"  - Failed: {failed}")
            print(f"  - Skipped: {skipped}")
    
    # Overall statistics
    print("\n" + "="*60)
    monitor_processing(all_files_processed)
    
    print(f"\nProcessed files DataFrame:")
    display(all_files_processed) if 'display' in globals() else print(all_files_processed)
else:
    print("No processing results found. Please run the processing cells above.")

In [ ]:
# Display final results
print(f"\n📊 Final Processing Results:")
print(f"Total files processed: {len(all_files_processed)}")
print(f"\nProcessed files DataFrame:")
all_files_processed

## Check STATUS
[Disasters Bucket](https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/)